# Phase 2 — all three arms on 13 tasks

primary 5 min/24 calls; probe 8 min/28 calls; calls40 8 min/40 calls. Same model, prompt, tasks.

## 1. Environment

In [3]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        'LiteLLM',
        'LiteLLM Router',
        'LiteLLM Proxy',
        'litellm',
        'httpx',
        'httpcore',
        'openai',
        'urllib3',
        'opentelemetry',
        'opentelemetry.context',
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')

Installing 41 wheels from /kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse...
Wheelhouse installation complete.


## 2. Load primary, create probe and calls40, select 13 tasks

In [4]:
from pathlib import Path
import re, shutil, zipfile

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')

SRC_PRIMARY = Path('/kaggle/input/datasets/utsavgupta1111/gemma4-primary')
if not SRC_PRIMARY.exists():
    hits = list(Path('/kaggle/input').rglob('gemma4-primary'))
    assert hits, 'gemma4-primary dataset not attached'
    SRC_PRIMARY = hits[0]

PRIMARY_DIR = WORKING_DIR / 'submission'
PROBE_DIR = WORKING_DIR / 'submission-probe'
CALLS40_DIR = WORKING_DIR / 'submission-calls40'

def extract_agent(src: Path, dst: Path) -> None:
    if dst.exists():
        shutil.rmtree(dst)
    dst.mkdir(parents=True)
    zips = list(src.rglob('*.zip'))
    if zips:
        print('extracting', zips[0])
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall(dst)
        return
    agents = list(src.rglob('agent.yaml'))
    assert agents, f'agent.yaml not found under {src}'
    root = agents[0].parent
    print('copying from', root)
    shutil.copytree(root, dst, dirs_exist_ok=True)

extract_agent(SRC_PRIMARY, PRIMARY_DIR)
assert (PRIMARY_DIR / 'agent.yaml').is_file()

def make_variant(dst: Path, calls: int, minutes: int, thinking: int, old_text: str, new_text: str) -> None:
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(PRIMARY_DIR, dst)
    (dst / 'eval_config.yaml').write_text(
        "evaluation:\n"
        "  timeout_seconds: 240\n"
        f"  max_tool_calls: {calls}\n"
        f"  max_time_minutes: {minutes}\n"
        "  max_turns: 80\n"
    )
    (dst / 'configs/sampling.yaml').write_text(
        "temperature: 0.2\n"
        "top_p: 0.95\n"
        "max_output_tokens: 8192\n"
        "thinking_config:\n"
        f"  thinking_budget: {thinking}\n"
        "  include_thoughts: true\n"
    )
    p = dst / 'prompts/system.md'
    p.write_text(p.read_text().replace(old_text, new_text))

make_variant(PROBE_DIR, 28, 8, 4096, '24 tool calls and 5 minutes', '28 tool calls and 8 minutes')
make_variant(CALLS40_DIR, 40, 8, 4096, '24 tool calls and 5 minutes', '40 tool calls and 8 minutes')

from swegemma.models import load_tasks
tasks = load_tasks(TASKS_PATH)
by_repo = {}
for t in tasks:
    by_repo.setdefault(t.repo, []).append(t)
SELECTED_TASKS = [t for ts in by_repo.values() for t in ts[:4]]

AGENT_DIR = PRIMARY_DIR  # vLLM model/adapters
print('primary:', sorted(x.name for x in PRIMARY_DIR.iterdir()))
print('probe  :', sorted(x.name for x in PROBE_DIR.iterdir()))
print('calls40:', sorted(x.name for x in CALLS40_DIR.iterdir()))
print('selected:', len(SELECTED_TASKS), [t.instance_id for t in SELECTED_TASKS])


copying from /kaggle/input/datasets/utsavgupta1111/gemma4-primary
primary: ['agent.yaml', 'configs', 'eval_config.yaml', 'prompts']
probe  : ['agent.yaml', 'configs', 'eval_config.yaml', 'prompts']
calls40: ['agent.yaml', 'configs', 'eval_config.yaml', 'prompts']
selected: 13 ['fastapi_15661', 'fastapi_15588', 'fastapi_15589', 'fastapi_15030', 'requests_7505', 'requests_7502', 'requests_7433', 'requests_7427', 'rich_4070', 'rich_4077', 'rich_4079', 'rich_4076', 'httpx_3672']


## 3. Start vLLM

In [5]:
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from adk_submission.discovery import discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

litellm.drop_params = True

ALLOWED_MODEL_NAMES = frozenset({
    'gemma-4-31b-it-qat-w4a16-ct',
    'gemma-4-12b-it-qat-w4a16-ct',
    'gemma-4-e4b-it-qat-w4a16-ct',
    'gemma-4-e2b-it-qat-w4a16-ct',
})
MODEL_VLLM_EXTRA_ARGS = {
    'gemma-4-12b-it-qat-w4a16-ct': (
        '--hf-overrides',
        '{"architectures": ["Gemma4ForCausalLM"], "model_type": "gemma4"}',
    ),
}
_SAFE_PATH_COMPONENT = re.compile(r'[A-Za-z0-9][A-Za-z0-9._-]*')
INFERENCE_API_KEY = os.environ.get('LOCAL_API_KEY', 'EMPTY')

# Validate single canonical base model and discover any PEFT LoRA adapters in the submission directory.
# Note: sample_submission includes dummy LoRA adapters (adapters/main_lora and adapters/tool_lora)
# shaped for gemma-4-31b-it-qat-w4a16-ct; Cell 2 removes them automatically when MODEL_NAME != 31B.
declared_models = discover_declared_models(AGENT_DIR)
assert len(declared_models) == 1, f'Expected exactly 1 declared model, got: {sorted(declared_models)}'
declared_model = next(iter(declared_models))
assert declared_model in ALLOWED_MODEL_NAMES, f'Unsupported model {declared_model!r}. Allowed: {sorted(ALLOWED_MODEL_NAMES)}'

MODEL_PATH = Path(f'/kaggle/input/models/google/gemma-4/other/{declared_model}/2')
assert MODEL_PATH.exists(), f'Model weights not found at {MODEL_PATH}. Ensure the model input is attached.'

adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
adapter_map = getattr(adapters, 'adapters', None) or {}
for adapter_name in adapter_map:
    assert _SAFE_PATH_COMPONENT.fullmatch(adapter_name), (
        f'Unsafe adapter name {adapter_name!r}: must match [A-Za-z0-9][A-Za-z0-9._-]*'
    )
detected_ranks = [
    r for info in adapter_map.values() if (r := getattr(info, 'rank', None)) is not None and r > 0
]

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

if 'server_instance' in globals() and server_instance is not None:
    server_instance.stop()

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=bool(adapter_map),
    max_loras=max(1, len(adapter_map)),
    max_lora_rank=max(detected_ranks) if detected_ranks else 128,
    scheduler_reserve_full_isl=False,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
    extra_args=MODEL_VLLM_EXTRA_ARGS.get(declared_model, ()),
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

# Register model and LoRA adapter aliases for Google ADK
models = server_instance.create_model_registry(
    aliases=[declared_model],
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)

vLLM server started on http://127.0.0.1:8000/v1 (tp=4)


## 4. Run primary, probe, calls40

In [6]:
import asyncio
import concurrent.futures
import pandas as pd
import yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator


def run_sync(coro_or_fn, *args, **kwargs):
    if callable(coro_or_fn) and not asyncio.iscoroutine(coro_or_fn):
        fn = lambda: coro_or_fn(*args, **kwargs)
    else:
        fn = lambda: coro_or_fn
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())


def evaluate_bundle(tag: str, agent_dir: Path) -> pd.DataFrame:
    eval_config_file = next(
        (c for name in ('eval_config.yaml', 'eval_config.yml') if (c := agent_dir / name).is_file()),
        None,
    )
    raw = yaml.safe_load(eval_config_file.read_text(encoding='utf-8')) or {} if eval_config_file else {}
    sec = raw.get('evaluation', raw)
    timeout_seconds = int(sec.get('timeout_seconds', 300))
    verification_timeout_seconds = int(sec.get('verification_timeout_seconds', 300))
    max_tool_calls = int(sec.get('max_tool_calls', 100))
    max_time_minutes = float(sec.get('max_time_minutes', 60.0))
    turns_raw = sec.get('max_turns', sec.get('max_llm_calls'))
    max_turns = int(turns_raw) if turns_raw is not None else None

    limits, gen_constraints = build_submission_limits()
    cfg = EvalConfig(
        tasks_path=TASKS_PATH,
        snapshots_dir=DATA_DIR / 'snapshots',
        results_dir=WORKING_DIR / f'results-{tag}',
        submission_dir=agent_dir,
        models=models,
        sandbox='subprocess',
        timeout_seconds=timeout_seconds,
        verification_timeout_seconds=verification_timeout_seconds,
        max_time_minutes=max_time_minutes,
        max_tool_calls=max_tool_calls,
        max_turns=max_turns,
        limits=limits,
        generation_constraints=gen_constraints,
        adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(
            compaction_interval=15,
            overlap_size=2,
            token_threshold=14336,
            event_retention_size=5,
        ),
        graph_dir=GRAPH_DIR,
        embeddings_dir=EMBEDDINGS_DIR,
        wheels_dir=DATA_DIR / 'wheels',
        verbose=False,
    )
    evaluator = Evaluator(cfg)
    rows = []
    for idx, task in enumerate(SELECTED_TASKS, start=1):
        print(f'[{tag} {idx}/{len(SELECTED_TASKS)}] {task.instance_id} ({task.repo})...')
        result = run_sync(evaluator.evaluate_task, task=task, task_index=idx, total_tasks=len(SELECTED_TASKS))
        patch = (result.agent_patch or '').strip()
        rows.append({
            'id': task.instance_id,
            'repo': task.repo,
            'prediction': patch if patch else 'NO_PATCH',
            'resolved': bool(result.resolved),
            'exit_code': result.test_exit_code,
            'patch_chars': len(patch),
            'tool_calls': result.tool_calls,
            'duration_s': result.duration_seconds,
        })
        print(f'  -> resolved={result.resolved}, patch_chars={len(patch)}, tool_calls={result.tool_calls}, duration={result.duration_seconds:.1f}s')

    df = pd.DataFrame(rows)
    df.to_csv(WORKING_DIR / f'task-results-{tag}.csv', index=False)
    df[['id', 'prediction']].to_csv(WORKING_DIR / f'predictions-{tag}.csv', index=False)
    return df


primary = evaluate_bundle('primary', PRIMARY_DIR)
probe = evaluate_bundle('probe', PROBE_DIR)
calls40 = evaluate_bundle('calls40', CALLS40_DIR)


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.AGENT_CONFIG is enabled.
  check_feature_enabled()
/tmp/ipykernel_163/1755210882.py:57: UserWarning: [EXPERIMENTAL] EventsCompactionConfig: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  events_compaction_config=EventsCompactionConfig(


[primary 1/13] fastapi_15661 (fastapi/fastapi)...


[   0.0s] 📋 System Prompt: You are an autonomous software engineer. Your job: make the smallest correct change 
to the SOURCE code in /workspace so ...
  [   0.0s] 🎯 Task Prompt: You are evaluating a software engineering task for repository fastapi/fastapi.
  [  27.7s] 💭 (2519 chars) "The user wants to "Automate release preparation".
Looking at the provided PR description, it mentions it's based on `htt..."
  [  27.7s] 🔧 run_command git grep -i "release"
  [  27.8s]   → .github/workflows/latest-changes.yml:          latest_changes_file: docs/en/docs/release-notes.md
  [  33.8s] 💭 (503 chars) "The grep results show `scripts/add_latest_release_date.py` and 
`.github/workflows/latest-changes.yml` which references `..."
  [  33.8s] 📖 read_file (scripts/add_latest_release_date.py)
  [  33.8s]   ✓ {'status': 'ok', 'filepath': 'scripts/add_latest_release_date.py', 'content': '"""Check 
release-notes.md and add today\'…
  [  40.6s] 💭 (699 chars) "This script just adds the date to the latest release header in `release-notes.md`.

The issue is "Automate release prepa..."
  [  40.6s] 📖 read_file (scripts/docs.py)
  [  40.6s]   ✓ {'status': 'ok', 'filepath': 'scripts/docs.py', 'content': 'import json\nimport logging\nimport 
os\nimport re\nimport sh…
  [  52.4s] 💭 (1239 chars) "`scripts/docs.py` seems to be for managing documentation translations.

Let's go back to the "Automate release preparati..."
  [  52.4s] 🔧 run_command grep -r "release" scripts/
  [  52.5s]   → scripts/add_latest_release_date.py:"""Check release-notes.md and add today's date to the latest 
release header if missin
  [  56.5s] 💭 (335 chars) "Still nothing obvious.

Wait, the prompt says "Codex with GPT-5.5, based on https://github.com/fastapi/asyncer/pull/608,..."
  [  56.5s] 🔧 run_command git grep -i "prepare"
  [  56.5s]   → docs/en/docs/async.md:The cashier says something to the cook in the kitchen so they know they have 
to prepare your burge
  [  58.1s] 💭 (68 chars) "Nothing useful.

Let's try to search for "automation" or "automate"."
  [  58.1s] 🔧 run_command git grep -i "automate"
  [  58.1s]   → docs/en/docs/advanced/events.md:You could load it at the top level of the module/file, but that 
would also mean that it 
  [  68.7s] 💭 (1146 chars) "Still nothing.

Wait, I might have missed something. Let's look at the `asyncer` PR 608 if I can. I can't.
But wait, if ..."
  [  68.7s] 📖 read_file (pyproject.toml)
  [  68.7s]   ✓ {'status': 'ok', 'filepath': 'pyproject.toml', 'content': '[build-system]\nrequires = 
["pdm-backend"]\nbuild-backend = "…
  [  88.7s] 💭 (1968 chars) "The version is dynamic: `dynamic = ["version"]`.

Wait, I just had a thought. If the task is "Automate release preparati..."
  [  88.7s] 📖 read_file (docs/en/docs/release-notes.md)
  [  88.8s]   ✓ {'status': 'ok', 'filepath': 'docs/en/docs/release-notes.md', 'content': '---\nhide:\n  - 
navigation\n---\n\n# Release N…
  [ 103.6s] 💭 (1285 chars) "The release notes show that they have a versioning system like `0.136.3 (2026-05-23)`.

Wait, I just noticed something. ..."
  [ 103.6s] 🔧 run_command find . -name "*release*"
  [ 103.6s]   → ./scripts/add_latest_release_date.py
  [ 130.8s] 💭 (273 chars) "Okay, so only those two.

Wait, maybe the "Automate release preparation" is about adding a GitHub Action?
But the prompt..."
  [ 130.8s] 🔧 run_command ls -R .github/workflows/
  [ 130.8s]   → .github/workflows/:
  [ 132.2s] 💭 (38 chars) "I see `publish.yml`. Let's look at it."
  [ 132.2s] 📖 read_file (.github/workflows/publish.yml)
  [ 132.2s]   ✓ {'status': 'ok', 'filepath': '.github/workflows/publish.yml', 'content': 'name: Publish\n\non:\n  
release:\n    types:\n…
  [ 141.1s] 💭 (1029 chars) "This is a simple publish workflow.

Wait, I just had a thought. If the task is "Automate release preparation", and it's ..."
  [ 141.1s] 📖 read_file (scripts/add_latest_release_date.py)
  [ 141.2s]   ✓ {'status': 'ok', 'filepath': 'scripts/add_latest_release_date.py', 'content': '"""Check 
release-notes.md an

  -> resolved=False, patch_chars=0, tool_calls=21, duration=305.1s
[primary 2/13] fastapi_15588 (fastapi/fastapi)...


[   0.0s] 📋 System Prompt: You are an autonomous software engineer. Your job: make the smallest correct change 
to the SOURCE code in /workspace so ...
  [   0.0s] 🎯 Task Prompt: You are evaluating a software engineering task for repository fastapi/fastapi.
  [   6.7s] 💭 (703 chars) "The user wants to validate Server Sent Event (SSE) fields to prevent sending broken 
data.
Specifically, if `id` or `even..."
  [   6.7s] 📖 read_file (fastapi/sse.py)
  [   6.8s]   ✓ {'status': 'ok', 'filepath': 'fastapi/sse.py', 'content': 'from typing import Annotated, 
Any\n\nfrom annotated_doc impor…
  [  64.9s] 💭 (7279 chars) "The `ServerSentEvent` class is a Pydantic model.
It has `event` and `id` fields.
`id` already has an `AfterValidator(_ch..."
  [  64.9s] 🔧 run_command grep -r "ServerSentEvent" .
  [  65.1s]   → ./fastapi/.agents/skills/fastapi/references/streaming.md:For full control over SSE fields (`event`,
`id`, `retry`, `comm
  [  69.6s] 💭 (308 chars) "The `ServerSentEvent` is a Pydantic model. When it's instantiated, Pydantic runs the 
validators.

If I add the newline c..."
  [  69.6s] 📖 read_file (tests/test_sse.py)
  [  69.6s]   ✓ {'status': 'ok', 'filepath': 'tests/test_sse.py', 'content': 'import asyncio\nimport time\nfrom 
collections.abc import A…
  [  73.0s] 💭 (145 chars) "I see some tests for `ServerSentEvent` validation at the end of the file (based on the 
`grep` output). Let's read the en..."
  [  73.0s] 📖 read_file (tests/test_sse.py)
  [  73.0s]   ✓ {'status': 'ok', 'filepath': 'tests/test_sse.py', 'content': '\n\ndef 
test_data_and_raw_data_mutually_exclusive():\n    …
  [  76.3s] 💭 (143 chars) "I missed some tests in my previous `read_file` because I only read the first 150 lines 
and then from 250. Let's see what..."
  [  76.3s] 📖 read_file (tests/test_sse.py)
  [  76.3s]   ✓ {'status': 'ok', 'filepath': 'tests/test_sse.py', 'content': 'def 
test_sync_generator_no_annotation(client: TestClient):…
  [ 102.9s] 💭 (1934 chars) "Okay, I see `test_server_sent_event_null_id_rejected`. I should add similar tests for 
newlines in `event` and `id`.

Now..."
  [ 102.9s] ✏️ edit_file (fastapi/sse.py)
  [ 103.0s]   ✓ {'status': 'ok', 'filepath': 'fastapi/sse.py', 'occurrences': 1, 'strategy': 'exact', 'diff': '--- 
a/fastapi/sse.py\n+++…
  [ 112.1s] 💭 (153 chars) "Now I should verify the fix. I'll create a small test script to verify that 
`ServerSentEvent` raises `ValueError` when `..."
  [ 112.1s] 🔧 run_command (25 lines) python3 -c 'from fastapi.sse import ServerSentEvent; import pytest; 
  [ 112.8s]   → Success: event with newline raised 1 validation error for ServerSentEvent
  [ 141.8s] 💭 (69 chars) "The tests passed. Now I should check if `data` still allows newlines."
  [ 141.8s] 🔧 run_command (7 lines) python3 -c 'from fastapi.sse import ServerSentEvent; 
  [ 142.3s]   → Success: data with newline allowed
  [ 144.3s] 💭 (115 chars) "Everything looks good. I'll now run the existing tests in `tests/test_sse.py` to make 
sure I didn't break anything."
  [ 144.3s] 🔧 run_command timeout 150 python -m pytest tests/test_sse.py -x -q
  [ 146.7s]   → ..................                                                       [100%]
  [ 147.5s] 💭 (38 chars) "All tests passed. I'm ready to submit."
  [ 147.5s] ✅ Agent finished: <|tool_call>call:git diff<|"|><tool_call|>
  [ 147.5s] 🎯 Task Prompt: Your previous response reached the token limit before the tool call finished closing 
(<|tool_call|> was cut off). Do NOT...
  [ 149.7s] 🤖 submit_patch
  [ 149.8s]   → submit_patch: {"status": "ok", "patch_size": 649, "files_changed": 1}
  [ 152.6s] ✅ Agent finished: The `ServerSentEvent` Pydantic model in `fastapi/sse.py` was updated to validate 
that the `id` and `event` fields do not contain multi-line content (newlines `\n` or carriage returns `\r`), 
raising a ...

╭───────────────────────────────────────────────── Latest Detail ─────────────────────────────────────────────────╮
│   ✅ Agent Finished                                      

...

  -> resolved=False, patch_chars=0, tool_calls=40, duration=365.6s
[calls40 12/13] rich_4076 (Textualize/rich)...


[   0.0s] 📋 System Prompt: You are an autonomous software engineer. Your job: make the smallest correct change 
to the SOURCE code in /workspace so ...
  [   0.0s] 🎯 Task Prompt: You are evaluating a software engineering task for repository Textualize/rich.
  [  10.0s] 💭 (1147 chars) "The problem statement is "preserve newlines" and refers to issue #3577.
I need to find where newlines are being lost or ..."
  [  10.0s] 🔧 run_command grep -n "newline" rich/_wrap.py
  [  10.0s]   → error (exit 0): 
  [  12.9s] 💭 (230 chars) "Okay, `grep` failed to find "newline" in `rich/_wrap.py`. Let's try "wrap" or "split".
Wait, maybe it's in `rich/text.py..."
  [  12.9s] 🔧 run_command grep -rn "preserve_newlines" rich/
  [  13.0s]   → error (exit 0): 
  [  14.4s] 💭 (94 chars) "No "preserve_newlines".

Let's look at `rich/_wrap.py` content to see how it handles wrapping."
  [  14.4s] 📖 read_file (rich/_wrap.py)
  [  14.4s]   ✓ {'status': 'ok', 'filepath': 'rich/_wrap.py', 'content': 'from __future__ import 
annotations\n\nimport re\nfrom typing i…
  [  30.6s] 💭 (1567 chars) "In `rich/_wrap.py`, `divide_line` uses `words(text)` to find where to break the line.
`words(text)` uses `re_word = re.c..."
  [  30.6s] 🔧 run_command python3 -c "from rich.console import Console; console = Console(width=20); conso
  [  30.8s]   → Hello
  [  33.7s] 💭 (245 chars) "Wait, `console.print('Hello\nWorld')` printed it on two lines.
Why? Because `console.print` probably handles newlines be..."
  [  33.7s] 🔧 run_command grep -n "print" rich/console.py | head -n 20
  [  33.7s]   → 311:    """Context manager to capture the result of printing to the console.
  [  35.4s] 💭 (55 chars) "I need to find the `print` method in `rich/console.py`."
  [  35.4s] 🔧 run_command grep -n "def print(" rich/console.py
  [  35.4s]   → 1652:    def print(
  [  37.2s] 💭 (51 chars) "Let's read the `print` method in `rich/console.py`."
  [  37.2s] 📖 read_file (rich/console.py)
  [  37.2s]   ✓ {'status': 'ok', 'filepath': 'rich/console.py', 'content': '    def print(\n        self,\n        
*objects: Any,\n     …
  [  40.6s] 💭 (183 chars) "The `print` method calls `self.render(renderable, render_options)`.
If `renderable` is a string, `self.render` will prob..."
  [  40.6s] 🔧 run_command grep -n "def render(" rich/console.py
  [  40.6s]   → 1294:    def render(
  [  42.3s] 💭 (41 chars) "Let's read `render` in `rich/console.py`."
  [  42.3s] 📖 read_file (rich/console.py)
  [  42.3s]   ✓ {'status': 'ok', 'filepath': 'rich/console.py', 'content': '    def render(\n        self, 
renderable: RenderableType, o…
  [  44.3s] 💭 (56 chars) "If `renderable` is a string, it calls `self.render_str`."
  [  44.3s] 🔧 run_command grep -n "def render_str(" rich/console.py
  [  44.3s]   → 1409:    def render_str(
  [  46.1s] 💭 (45 chars) "Let's read `render_str` in `rich/console.py`."
  [  46.1s] 📖 read_file (rich/console.py)
  [  46.1s]   ✓ {'status': 'ok', 'filepath': 'rich/console.py', 'content': '    def render_str(\n        self,\n   
text: str,\n    …
  [  48.2s] 💭 (65 chars) "It returns a `Text` instance. Let's see the rest of `render_str`."
  [  48.2s] 📖 read_file (rich/console.py)
  [  48.2s]   ✓ {'status': 'ok', 'filepath': 'rich/console.py', 'content': '            highlight (Optional[bool], 
optional): Enable hig…
  [  50.9s] 💭 (145 chars) "So it creates a `Text` object. Now let's see how `Text` is rendered.
`Text` probably has a `__rich_console__` method. Le..."
  [  50.9s] 🔧 run_command grep -n "def __rich_console__" rich/text.py
  [  50.9s]   → 689:    def __rich_console__(
  [  52.7s] 💭 (53 chars) "Let's read `Text.__rich_console__` in `rich/text.py`."
  [  52.7s] 📖 read_file (rich/text.py)
  [  52.8s]   ✓ {'status': 'ok', 'filepath': 'rich/text.py', 'content': '    def __rich_console__(\n        self, 
console: "Console", op…
  [  54.9s] 💭 (66 chars) "`Text.__rich_console__` calls `self.wrap`. Let's find `Text.wrap`."
  [  54.9s] 🔧 run_command grep -n "def wrap(" rich/text.py
  [  55.0s] 

  -> resolved=False, patch_chars=0, tool_calls=40, duration=480.9s
[calls40 13/13] httpx_3672 (encode/httpx)...


[   0.0s] 📋 System Prompt: You are an autonomous software engineer. Your job: make the smallest correct change 
to the SOURCE code in /workspace so ...
  [   0.0s] 🎯 Task Prompt: You are evaluating a software engineering task for repository encode/httpx.
  [   7.0s] 💭 (755 chars) "The problem statement is a bit sparse, but it lists several specific requirements 
related to server connection handling ..."
  [   7.0s] 🔧 run_command grep -n "class HTTPParser" src/httpx/_parsers.py
  [   7.0s]   → 72:class HTTPParser:
  [   8.9s] 💭 (111 chars) "Okay, `HTTPParser` is in `src/httpx/_parsers.py` starting at line 72.
Now let's look at `src/httpx/_server.py`."
  [   8.9s] 🔧 run_command ls src/httpx/_server.py
  [   8.9s]   → src/httpx/_server.py
  [  10.4s] 💭 (54 chars) "Let's examine `HTTPParser` in `src/httpx/_parsers.py`."
  [  10.4s] 📖 read_file (src/httpx/_parsers.py)
  [  10.5s]   ✓ {'status': 'ok', 'filepath': 'src/httpx/_parsers.py', 'content': 'class HTTPParser:\n    """\n    
Usage...\n\n    client…
  [  13.1s] 💭 (73 chars) "I need to see the rest of `HTTPParser`, especially the `complete` method."
  [  13.1s] 📖 read_file (src/httpx/_parsers.py)
  [  13.1s]   ✓ {'status': 'ok', 'filepath': 'src/httpx/_parsers.py', 'content': '\n        if body == b\'\':\n    
# Handle body…
  [  15.6s] 💭 (33 chars) "Still haven't reached `complete`."
  [  15.6s] 📖 read_file (src/httpx/_parsers.py)
  [  15.6s]   ✓ {'status': 'ok', 'filepath': 'src/httpx/_parsers.py', 'content': '\n        if body == b\'\':\n    
# Handle body…
  [  29.8s] 💭 (1448 chars) "Okay, I see `complete` method:

```python
    def complete(self):
        is_fully_complete = self.send_state == State.D..."
  [  29.8s] 📖 read_file (src/httpx/_server.py)
  [  29.8s]   ✓ {'status': 'ok', 'filepath': 'src/httpx/_server.py', 'content': 'import contextlib\nimport 
logging\nimport time\n\nfrom …
  [  49.9s] 💭 (2602 chars) "I see a bug in `HTTPConnection._complete`:
```python
    def _complete(self):
        self._parser.complete
        self..."
  [  49.9s] 📖 read_file (src/httpx/_streams.py)
  [  49.9s]   ✓ {'status': 'ok', 'filepath': 'src/httpx/_streams.py', 'content': 'import io\nimport typing\nimport 
types\nimport os\n\n\…
  [  96.1s] 💭 (5646 chars) "In `HTTPStream.close()`:
```python
    def close(self) -> None:\n        self._buffer.close()\n        if self._complete..."
  [  96.1s] 📖 read_file (src/httpx/_network.py)
  [  96.2s]   ✓ {'status': 'ok', 'filepath': 'src/httpx/_network.py', 'content': 'import concurrent.futures\nimport
contextlib\nimport c…
  [  99.1s] 💭 (92 chars) "I need to see the rest of `NetworkBackend` and `NetworkListener` in 
`src/httpx/_network.py`."
  [  99.1s] 📖 read_file (src/httpx/_network.py)
  [  99.1s]   ✓ {'status': 'ok', 'filepath': 'src/httpx/_network.py', 'content': '        traceback: 
types.TracebackType | None = None,\…
  [ 148.3s] 💭 (3419 chars) "In `NetworkServer.__exit__`:
```python
    def __exit__(self, exc_type, exc_val, exc_tb):\n        self.listener.close()..."
  [ 148.3s] 📖 read_file (src/httpx/_parsers.py)
  [ 148.3s]   ✓ {'status': 'ok', 'filepath': 'src/httpx/_parsers.py', 'content': 'import enum\n\nfrom ._streams 
import Stream\n\n__all__…
  [ 151.1s] 💭 (66 chars) "I need to see the rest of `HTTPParser` in `src/httpx/_parsers.py`."
  [ 151.1s] 📖 read_file (src/httpx/_parsers.py)
  [ 151.1s]   ✓ {'status': 'ok', 'filepath': 'src/httpx/_parsers.py', 'content': '\n        self.send_state = 
State.SEND_HEADERS\n\n    …
  [ 154.1s] 💭 (65 chars) "I need to see the end of `HTTPParser` in `src/httpx/_parsers.py`."
  [ 154.1s] 📖 read_file (src/httpx/_parsers.py)
  [ 154.2s]   ✓ {'status': 'ok', 'filepath': 'src/httpx/_parsers.py', 'content': '\n    def close(self):\n        
if self.send_state != …
  [ 156.6s] 💭 (59 chars) "I missed `complete` method in `HTTPParser`. Let me find it."
  [ 156.6s] 🔧 run_command grep -n "def complete" src/httpx/_parsers.py
  [ 156.6s]   → 378:    def complete(self):
  [ 194.0s] 💭 (49 chars) "Okay, it'

  -> resolved=False, patch_chars=2340, tool_calls=39, duration=344.5s


## 5. Summary

In [7]:
import pandas as pd
from pathlib import Path

all_rows = []
for name, df in [('primary', primary), ('probe', probe), ('calls40', calls40)]:
    print(f'\n=== {name} ===')
    print('resolved:', int(df.resolved.sum()), '/', len(df))
    print(df.groupby('repo').resolved.agg(['sum', 'count']))
    print('non-empty patches:', int((df.prediction != 'NO_PATCH').sum()), '/', len(df))
    print('duration p50/p95:', round(df.duration_s.quantile(.5), 1), round(df.duration_s.quantile(.95), 1))
    print('mean tool calls:', round(df.tool_calls.mean(), 1), 'max:', int(df.tool_calls.max()))
    print('hit call cap:', int((df.tool_calls >= df.tool_calls.max()).sum()))
    print('NO_PATCH:', df.loc[df.prediction == 'NO_PATCH', 'id'].tolist())
    d = df.copy(); d['arm'] = name; all_rows.append(d)

combined = pd.concat(all_rows, ignore_index=True)
combined.to_csv('/kaggle/working/task-results-all.csv', index=False)
print('\nwrote /kaggle/working/task-results-all.csv')

print('\n=== summary ===')
for name, df in [('primary', primary), ('probe', probe), ('calls40', calls40)]:
    print(name, 'resolved', int(df.resolved.sum()), 'patches', int((df.prediction != 'NO_PATCH').sum()),
          'p95_s', round(df.duration_s.quantile(.95), 1),
          'projected_120h', round(df.duration_s.mean() * 120 / 3600, 1))



=== primary ===
resolved: 0 / 13
                 sum  count
repo                       
Textualize/rich    0      4
encode/httpx       0      1
fastapi/fastapi    0      4
psf/requests       0      4
non-empty patches: 7 / 13
duration p50/p95: 305.1 352.3
mean tool calls: 21.8 max: 24
hit call cap: 9
NO_PATCH: ['fastapi_15661', 'fastapi_15589', 'requests_7502', 'requests_7433', 'rich_4077', 'rich_4076']

=== probe ===
resolved: 2 / 13
                 sum  count
repo                       
Textualize/rich    1      4
encode/httpx       0      1
fastapi/fastapi    1      4
psf/requests       0      4
non-empty patches: 12 / 13
duration p50/p95: 339.2 478.0
mean tool calls: 24.7 max: 28
hit call cap: 8
NO_PATCH: ['rich_4079']

=== calls40 ===
resolved: 1 / 13
                 sum  count
repo                       
Textualize/rich    1      4
encode/httpx       0      1
fastapi/fastapi    0      4
psf/requests       0      4
non-empty patches: 10 / 13
duration p50/p95: 373.1 488.4
mean 

## 6. Stop server

In [8]:
server_instance.stop()
print("vLLM stopped")

vLLM stopped
